# Deliverable A — Calibrated Default Probability

**Goal:** For every applicant in `validation.csv` and `test.csv` (13,306 total), output:
- `decision` (0=decline, 1=approve)
- `predicted_pd` — calibrated probability of default
- `pd_lower_90`, `pd_upper_90` — 90 % prediction interval

**Decision rule:** Approve when `predicted_pd < breakeven_pd_i`, where the threshold is
**per-applicant** — derived from the full NPV cash-flow model and each borrower's origination
segment (credit band × sector) via the CDR matrix.

## Pipeline overview

| Step | What | Output |
|------|------|--------|
| 1 | Load & split data | `train`, `val_all`, `val_labeled` |
| 2 | Integrity checks | Planted violation counts |
| 3 | Feature engineering | 47 features incl. MNAR indicators |
| 4 | Build matrices | numpy arrays for CatBoost |
| 5 | 10-fold CV *(diagnostic)* | OOF AUC — model quality estimate only |
| 6 | Ensemble final model + bootstrap | `boot` dict: calibrated PDs + intervals |
| 7 | Calibration metrics | AUC, Logloss, Brier on `val_labeled` |
| 8 | CDR matrix + per-loan thresholds | `breakeven_pds` array (one per applicant) |
| 9 | Assemble submission | `submission_A_decisions.csv` |
| 10 | Format validation | Pass/fail vs. validator script |

## Known traps handled

| Trap | Mitigation |
|------|------------|
| Outcome leakage | `OUTCOME_COLS` excluded before any fit |
| `prior_decision` collider | Excluded from feature set |
| MNAR bank-feed nulls | `{col}__missing` indicators added for all 6 bank-feed columns |
| Reject inference | Acknowledged; partially mitigated via `prior_underwriter_score` + val calibration |
| Single-model variance | 5-member `CatBoostEnsemble` (varied seeds + holdout splits) used as base |
| Probability calibration | Isotonic regression per bootstrap model on held-out `val_labeled` |
| Prediction intervals | Bootstrap [5th, 95th] percentile; conformal correction if coverage < 90 % |
| Break-even threshold | Full NPV formula (partial repayments + recovery); per-applicant via CDR matrix |

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'code')   # make code/ importable from repo root

import numpy as np
import pandas as pd

# ── Module imports (all logic lives in code/) ─────────────────────────────────
from config      import SEED, DATA_DIR, SUB_DIR, N_FOLDS
from data        import load_raw, prepare_splits, engineer_features, build_feature_cols, build_matrices
from integrity   import check_integrity, check_split_leakage
from model       import run_cv, train_final_model
from calibration import bootstrap_calibrated_intervals
from decision    import (compute_breakeven_pd, compute_cdr_matrix,
                         compute_per_loan_breakeven, make_decisions)
from submission  import build_submission_a

# sklearn metrics used inline in this notebook for display
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

np.random.seed(SEED)
print('Imports OK')

Imports OK


## 1. Load data

In [2]:
train_raw, val_raw, test = load_raw()

# train: drop missing default_flag (declined/immature) upfront
# val_all: all rows → submission | val_labeled: known outcomes → calibration & optimisation
train, val_all, val_labeled, val_labeled_positions = prepare_splits(train_raw, val_raw)

print(f'train (labeled) : {len(train):>6,}  default rate {train["default_flag"].mean():.4f}')
print(f'val_all         : {len(val_all):>6,}  → submission')
print(f'val_labeled     : {len(val_labeled):>6,}  → calibration / optimisation')
print(f'test            : {len(test):>6,}')

train (labeled) : 51,722  default rate 0.1745
val_all         :  4,489  → submission
val_labeled     :  2,551  → calibration / optimisation
test            :  8,817


## 2. Integrity checks (known planted violations)

In [3]:
check_integrity(train, 'train (labeled)')
print()
check_split_leakage(train, val_all, test)

--- Integrity: train (labeled) ---
  prior_loans_default_count > prior_loans_count: 0
  days_to_default out of [1, 90]: 0
  default_flag / repayment_status mismatch: 0

--- Split leakage (business_id) ---
  train ∩ val  : 0  (expected 0)
  train ∩ test : 0  (expected 0)
  val   ∩ test : 0  (expected 0)


## 3. Feature engineering

In [4]:
train_fe       = engineer_features(train)
val_all_fe     = engineer_features(val_all)
val_labeled_fe = engineer_features(val_labeled)
test_fe        = engineer_features(test)

feature_cols, cat_indices = build_feature_cols(train_fe)
print(f'{len(feature_cols)} features,  {len(cat_indices)} categorical: {[feature_cols[i] for i in cat_indices]}')

47 features,  6 categorical: ['sector', 'geography_region', 'employee_count_bucket', 'intended_use_of_funds', 'owner_personal_credit_band', 'application_channel']


## 4. Prepare feature matrices

`train` is already filtered to labeled rows only (NaN `default_flag` dropped in `prepare_splits`).

`val_labeled_fe` is used for isotonic calibration and conformal coverage checks — it is
completely independent of training (no leakage).  `val_all_fe` (all 4,489 val rows including
unlabeled) feeds the submission predictions.

**Reject inference note:** The model is trained only on approved + matured loans.
Applicants the prior lender declined have no observed outcome, so the model may
under-estimate PD for profiles that were historically rejected. This bias is partially
mitigated by including `prior_underwriter_score` and `has_prior_approval` as features,
and by calibrating on the independent `val_labeled` set.

In [5]:
arrs = build_matrices(train_fe, val_all_fe, val_labeled_fe, test_fe, feature_cols)

print(f'X_train       : {arrs["X_train"].shape}')
print(f'X_val_labeled : {arrs["X_val_labeled"].shape}  (y_val_labeled default rate {arrs["y_val_labeled"].mean():.4f})')
print(f'X_val_all     : {arrs["X_val_all"].shape}')
print(f'X_test        : {arrs["X_test"].shape}')

X_train       : (51722, 47)
X_val_labeled : (2551, 47)  (y_val_labeled default rate 0.2062)
X_val_all     : (4489, 47)
X_test        : (8817, 47)


## 5. 10-fold stratified CatBoost CV *(diagnostic only)*

This step estimates model quality via out-of-fold (OOF) AUC. It does **not** produce the
submission predictions — those come from the bootstrap pipeline in Step 6.

Running CV here lets us detect overfitting (large gap between fold AUC and OOF AUC)
and compare against future experiments without touching the submission path.

In [6]:
# Diagnostic — OOF AUC only. Submission predictions come from train_final_model + bootstrap.
oof_preds, val_preds_folds, test_preds_folds, models = run_cv(
    arrs['X_train'], arrs['y_train'],
    arrs['X_val_all'], arrs['X_test'],
    cat_indices,
)

  Fold  1/10  AUC=0.7734  trees=216
  Fold  2/10  AUC=0.7728  trees=125
  Fold  3/10  AUC=0.7719  trees=114
  Fold  4/10  AUC=0.7728  trees=186
  Fold  5/10  AUC=0.7869  trees=126
  Fold  6/10  AUC=0.7685  trees=212
  Fold  7/10  AUC=0.7835  trees=268
  Fold  8/10  AUC=0.7741  trees=201
  Fold  9/10  AUC=0.7731  trees=142
  Fold 10/10  AUC=0.7822  trees=164

  OOF AUC  : 0.7758
  Fold mean: 0.7759 ± 0.0057


## 6. Ensemble final model + bootstrap calibrated intervals

**Why a K-fold ensemble base model?**
A single full-data fit can overfit because there is no held-out set providing
regularisation pressure.  With `ensemble=True`, `train_final_model` runs a 10-fold
stratified split: each member trains on a **different** 9/10 of the data, using the
remaining 1/10 as its early-stopping eval set.  Predictions are averaged across all 10
members, giving structural diversity (different training subsets) rather than just
random-state diversity.

**To revert to a single model:** pass `ensemble=False` to `train_final_model`.

**Bootstrap pipeline (steps 2–4):**
1. Draw 200 bootstrap resamples of `X_train`; train a fresh CatBoost on each.
2. Calibrate each bootstrap model's raw val predictions with isotonic regression (fitted on `val_labeled`); apply the calibrator to `val_all` and `test`.
3. `predicted_pd` = bootstrap mean; `[pd_lower_90, pd_upper_90]` = [5th, 95th] percentile.
4. Conformal coverage check on `val_labeled`: if empirical coverage < 90 %, expand intervals by the nonconformity quantile `q_90`.

In [7]:
# Step 1: train a 10-fold stratified ensemble.
# Each of the 10 members trains on a different 9/10 of the data; the held-out
# fold provides the early-stopping eval set, so no data is wasted permanently.
# Pass ensemble=False to revert to a single model.
final_model = train_final_model(
    arrs['X_train'], arrs['y_train'], cat_indices,
    ensemble=True, n_folds=10,
)

# Steps 2–4: bootstrap 200 models → per-bootstrap isotonic calibration →
#            [5th, 95th] percentile intervals → conformal coverage correction.
# final_model.predict_proba() averages across all 10 ensemble members transparently.
boot = bootstrap_calibrated_intervals(
    final_model,
    arrs['X_train'],    arrs['y_train'],
    arrs['X_val_all'],  arrs['y_val_labeled'], val_labeled_positions,
    arrs['X_test'],
    cat_indices,
)

  Fold  1/10  train=46,549  val=5,173  trees=216
  Fold  2/10  train=46,549  val=5,173  trees=131
  Fold  3/10  train=46,550  val=5,172  trees=156
  Fold  4/10  train=46,550  val=5,172  trees=114
  Fold  5/10  train=46,550  val=5,172  trees=179
  Fold  6/10  train=46,550  val=5,172  trees=138
  Fold  7/10  train=46,550  val=5,172  trees=159
  Fold  8/10  train=46,550  val=5,172  trees=231
  Fold  9/10  train=46,550  val=5,172  trees=169
  Fold 10/10  train=46,550  val=5,172  trees=136
Ensemble: 10 members, 114–231 trees (mean 163)


Bootstrap: 100%|██████████| 200/200 [04:53<00:00,  1.47s/model]


Bootstrap interval coverage on val: 6.4%  (target: 90%)
After conformal correction (q_90=0.7255): 91.5%


## 7. Calibration quality on `val_labeled`

Metrics are computed on the 2,551 labeled val rows — the same set used to fit the
isotonic calibrators. Because isotonic regression is fitted **per bootstrap model** on
each model's own raw predictions, the calibration is independent of training.

A well-calibrated model has `Mean PD ≈ actual default rate` and a low Brier score.

In [24]:
y_vl            = arrs['y_val_labeled']
val_cal_labeled = boot['val_cal_labeled']   # bootstrap-mean calibrated PD on labeled val

print(f'AUC     : {roc_auc_score(y_vl, val_cal_labeled):.4f}')
print(f'Logloss : {log_loss(y_vl, val_cal_labeled):.4f}')
print(f'Brier   : {brier_score_loss(y_vl, val_cal_labeled):.4f}')
print(f'Mean PD : {val_cal_labeled.mean():.4f}  (actual default rate {y_vl.mean():.4f})')
print()

# Sanity checks: intervals must contain the point estimate and stay within [0, 1]
assert (boot['val_lower']  <= boot['val_cal_all']).all()
assert (boot['val_cal_all']  <= boot['val_upper']).all()
assert (boot['test_lower'] <= boot['test_cal']).all()
assert (boot['test_cal']   <= boot['test_upper']).all()
assert np.all((boot['val_cal_all'] >= 0) & (boot['val_cal_all'] <= 1))
assert np.all((boot['test_cal']    >= 0) & (boot['test_cal']    <= 1))
print('Interval assertions passed.')
print(f'Bootstrap coverage on val_labeled : {boot["coverage"] * 100:.1f}%  (target ≥ 90%)')
print(f'Mean interval width — val_all     : {(boot["val_upper"]  - boot["val_lower"]).mean():.4f}')
print(f'Mean interval width — test        : {(boot["test_upper"] - boot["test_lower"]).mean():.4f}')

AUC     : 0.7669
Logloss : 0.4189
Brier   : 0.1317
Mean PD : 0.2062  (actual default rate 0.2062)

Interval assertions passed.
Bootstrap coverage on val_labeled : 91.5%  (target ≥ 90%)
Mean interval width — val_all     : 0.9290
Mean interval width — test        : 0.9290


## 8. Approval decision — CDR-optimised per-loan thresholds

**Why per-loan thresholds?**  
The break-even PD depends on how much cash is recovered *before* a borrower defaults (partial repayments `D × (t*−1)`). A borrower who defaults on day 55 of 60 has almost fully repaid; one who defaults on day 2 is a near-total loss. The CDR matrix captures the empirical distribution of `t*` per origination segment (`credit_band × sector`), giving each applicant a threshold calibrated to their segment's expected default timing.

**Steps:**
1. `compute_cdr_matrix` — from training defaults, build `P(t*=t | segment)` and `E[rec/R | segment]`
2. `compute_breakeven_pd` — global reference using precise NPV formula (F + partial repayments + recovery)
3. `compute_per_loan_breakeven` — per-applicant threshold using their segment's CDR statistics

In [25]:
# Global reference threshold (precise NPV formula)
global_breakeven, avg_npv_default, avg_npv_repay = compute_breakeven_pd(train_raw)
print(f'Global break-even PD : {global_breakeven:.4f}  ({global_breakeven * 100:.2f} %)')
print(f'  avg NPV | repay    : {avg_npv_repay:.4f}')
print(f'  avg NPV | default  : {avg_npv_default:.4f}')
print()

# CDR matrix: P(t* = t | default, credit_band × sector)
cdr_stats = compute_cdr_matrix(train_raw)
print()
print('E[t*] by segment (top 10 by segment size):')
print(cdr_stats['expected_t'].sort_values().to_string())
print()
print('E[rec/R] by segment:')
print(cdr_stats['expected_rec_rate'].round(4).to_string())

Global break-even PD : 0.2251  (22.51 %)
  avg NPV | repay    : 2028.6060
  avg NPV | default  : -6984.9583

CDR matrix: 25 segments × 60 days
Global E[t*]=36.3d  global E[rec/R]=0.0914

E[t*] by segment (top 10 by segment size):
owner_personal_credit_band  sector
0                           1         32.476483
                            4         32.904255
1                           0         33.326733
0                           0         33.388290
                            2         33.548315
1                           2         34.572840
2                           1         35.012579
1                           1         35.257042
                            4         35.779817
2                           0         36.045455
                            2         36.083130
0                           3         36.870968
3                           0         37.020258
2                           4         37.268293
3                           4         37.421687
4              

## 9. Build `submission_A_decisions.csv`

`val_all` and `test` rows are concatenated in that order to match the order
of `boot['val_cal_all']` / `boot['test_cal']` and the per-loan `breakeven_pds` array.

`make_decisions` accepts either a scalar threshold (global) or a per-row array
(CDR-optimised). The per-loan threshold varies by `credit_band × sector` based on
the segment's expected default timing and recovery rate from the CDR matrix.

In [26]:
# Per-applicant break-even thresholds (val_all rows first, then test rows)
all_applicants = pd.concat([val_all, test], ignore_index=True)
breakeven_pds  = compute_per_loan_breakeven(all_applicants, cdr_stats)

print(f'Per-loan break-even PD — mean: {breakeven_pds.mean():.4f}  '
      f'std: {breakeven_pds.std():.4f}  '
      f'range: [{breakeven_pds.min():.4f}, {breakeven_pds.max():.4f}]')
print(f'Global reference threshold    : {global_breakeven:.4f}')
print()

all_pds   = np.concatenate([boot['val_cal_all'], boot['test_cal']])
lower     = np.concatenate([boot['val_lower'],   boot['test_lower']])
upper     = np.concatenate([boot['val_upper'],   boot['test_upper']])
decisions = make_decisions(all_pds, breakeven_pds)

submission, out_path = build_submission_a(
    val_all, test, boot['val_cal_all'], boot['test_cal'], decisions, lower, upper
)
print(f'Saved {len(submission):,} rows → {out_path}')
print(f'Approval rate  : {decisions.mean():.4f}  ({decisions.sum():,} approved)')
print(f'Mean PD        : {all_pds.mean():.4f}')
print(f'Mean width     : {(upper - lower).mean():.4f}')
print()
print(submission.describe())

Per-loan break-even PD — mean: 0.2736  std: 0.0626  range: [0.2084, 0.5256]
Global reference threshold    : 0.2251

Saved 13,306 rows → submissions/submission_A_decisions.csv
Approval rate  : 0.6214  (8,269 approved)
Mean PD        : 0.2765
Mean width     : 0.9290

           decision  predicted_pd   pd_lower_90   pd_upper_90
count  13306.000000  13306.000000  13306.000000  13306.000000
mean       0.621449      0.276500      0.005068      0.934041
std        0.485044      0.227565      0.034282      0.072205
min        0.000000      0.001583      0.000000      0.737179
25%        0.000000      0.121390      0.000000      0.871959
50%        1.000000      0.194785      0.000000      0.956914
75%        1.000000      0.368440      0.000000      1.000000
max        1.000000      1.000000      0.274510      1.000000


## 10. Validate submission format

Runs `validate_submission.py` against the `submissions/` directory.
Errors B and C are expected at this stage — Deliverables B and C are not yet built.
The A file should pass all A-specific checks (correct row count, columns, value ranges).

In [27]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'validate_submission.py', str(SUB_DIR)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Expecting 13,306 applicants, 900 queries, 13x13 trajectory grid.

ISSUES
------------------------------------------------------------------------------
  [ERROR] files / missing_or_unparsable: required file 'submission_B_trajectory.csv' is missing or not valid CSV
  [ERROR] files / missing_or_unparsable: required file 'submission_C_counterfactuals.csv' is missing or not valid CSV
  [warn ] files / missing_writeup: writeup 'submission_D_writeup.pdf' not found (include it in your real submission)
------------------------------------------------------------------------------

RESULT: FAIL  (2 error(s), 1 warning(s))
Fix every ERROR above and re-run. A failing submission cannot be scored.

STDERR: 
